# Create IDRC Awards from IATI Activity Files

Creates IDRC (International Development Research Centre) awards from IATI XML
activity files published on open.canada.ca. Canada-based funder with a Global
South research focus; activities span 2009 onward.

**Prerequisites:**
- Run `scripts/local/idrc_to_s3.py` to download IATI XMLs and upload the parquet first.

**Data source:** open.canada.ca CKAN package `idrc-crdi`, IATI Standard 2.03 XML files
**S3 location:** `s3a://openalex-ingest/awards/idrc/idrc_projects.parquet`

**IDRC funder (OpenAlex):**
- funder_id: 4320319949
- ROR: https://ror.org/0445x0472
- DOI: 10.13039/501100000193
- display_name: "International Development Research Centre"

**Schema notes:**
- Source rows preserve IATI sub-elements as JSON strings (`budgets_json`,
  `transactions_json`, `participating_orgs_json`, etc.). The transformation
  parses these via `from_json` and aggregates with higher-order functions.
- Amount = sum of all budget `value`s. Currency = first non-null currency from
  the budget list (consistently CAD across the IDRC corpus).
- Lead investigator affiliation = the first participating-org with role=4
  (implementing); IDRC funds organisations, not individuals, so given/family
  names are NULL.


## Step 1: Create staging table from S3

In [ ]:
%sql
-- Create the staging table from S3 parquet
CREATE OR REPLACE TABLE openalex.awards.idrc_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/idrc/idrc_projects.parquet`;

In [ ]:
%sql
-- Row count (smoke-test on 1 file showed 205; full corpus expected to be much larger)
SELECT COUNT(*) as total_activities FROM openalex.awards.idrc_raw;

## Step 1.5: Inspect raw data — verify columns before transforming
Per `plans/awards/how-to-add-a-funder.md` Step 1.5/1.6: confirm column names and
sanity-check amount/currency before writing the transformation.

In [ ]:
%sql
DESCRIBE openalex.awards.idrc_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.idrc_raw LIMIT 3;

In [ ]:
%sql
-- Sanity-check amount-shape and currency BEFORE writing the transformation
WITH parsed AS (
  SELECT from_json(budgets_json,
    'array<struct<status:string,type:string,value:string,currency:string>>') as budgets
  FROM openalex.awards.idrc_raw
)
SELECT
  COUNT(*) as rows,
  COUNT(budgets) as has_budgets,
  -- Sum every budget value per row, then aggregate stats across rows
  ROUND(MIN(AGGREGATE(budgets, 0.0D, (a, b) -> a + COALESCE(TRY_CAST(b.value AS DOUBLE), 0.0))), 0) as min_total,
  ROUND(MAX(AGGREGATE(budgets, 0.0D, (a, b) -> a + COALESCE(TRY_CAST(b.value AS DOUBLE), 0.0))), 0) as max_total,
  ROUND(AVG(AGGREGATE(budgets, 0.0D, (a, b) -> a + COALESCE(TRY_CAST(b.value AS DOUBLE), 0.0))), 0) as avg_total,
  -- Distinct currencies across all activities
  collect_set(try_element_at(FILTER(budgets, b -> b.currency IS NOT NULL), 1).currency) as currencies
FROM parsed;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.idrc_awards
USING delta
AS
WITH idrc_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320319949  -- International Development Research Centre
),
parsed AS (
    SELECT
        iati_identifier,
        title_en, title_fr, title_es,
        description_en, description_fr, description_es,
        actual_start, planned_start, actual_end, planned_end,
        activity_status_code,
        from_json(budgets_json,
            'array<struct<status:string,type:string,period_start:string,period_end:string,value:string,currency:string,value_date:string>>') as budgets,
        from_json(participating_orgs_json,
            'array<struct<role:string,type:string,ref:string,name_en:string,name_fr:string,name_es:string>>') as orgs,
        from_json(recipient_countries_json,
            'array<struct<code:string,percentage:string>>') as recipient_countries,
        source_xml_url
    FROM openalex.awards.idrc_raw
),
aggregated AS (
    SELECT
        iati_identifier,
        COALESCE(title_en, title_fr, title_es) as display_name,
        COALESCE(description_en, description_fr, description_es) as description,
        -- Total budget across all periods
        AGGREGATE(
            COALESCE(budgets, ARRAY()),
            0.0D,
            (acc, b) -> acc + COALESCE(TRY_CAST(b.value AS DOUBLE), 0.0)
        ) as total_amount,
        -- First non-null currency in the budget list
        try_element_at(FILTER(COALESCE(budgets, ARRAY()), b -> b.currency IS NOT NULL), 1).currency as currency,
        -- Implementing organisation (role=4) is the recipient
        try_element_at(FILTER(COALESCE(orgs, ARRAY()), o -> o.role = '4'), 1) as implementing_org,
        -- Recipient country code (ISO-3166)
        try_element_at(COALESCE(recipient_countries, ARRAY()), 1).code as recipient_country,
        -- Dates: prefer actual over planned
        COALESCE(TRY_TO_DATE(actual_start, 'yyyy-MM-dd'),
                 TRY_TO_DATE(planned_start, 'yyyy-MM-dd')) as start_date,
        COALESCE(TRY_TO_DATE(actual_end, 'yyyy-MM-dd'),
                 TRY_TO_DATE(planned_end, 'yyyy-MM-dd')) as end_date,
        activity_status_code,
        source_xml_url
    FROM parsed
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(a.iati_identifier)))) % 9000000000 as id,
    a.display_name,
    a.description,
    f.funder_id,
    a.iati_identifier as funder_award_id,
    -- Amount: only emit if > 0 (zero-budget activities are mostly admin / placeholders)
    CASE WHEN a.total_amount > 0 THEN a.total_amount ELSE NULL END as amount,
    a.currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'grant' as funding_type,
    -- IDRC IATI does not expose a programme/scheme code; leave NULL
    CAST(NULL AS STRING) as funder_scheme,
    'idrc_iati' as provenance,
    a.start_date,
    a.end_date,
    YEAR(a.start_date) as start_year,
    YEAR(a.end_date) as end_year,
    -- IDRC funds organisations; lead "investigator" affiliation = implementing org
    struct(
        CAST(NULL AS STRING) as given_name,
        CAST(NULL AS STRING) as family_name,
        CAST(NULL AS STRING) as orcid,
        CAST(NULL AS DATE) as role_start,
        struct(
            COALESCE(a.implementing_org.name_en, a.implementing_org.name_fr, a.implementing_org.name_es) as name,
            a.recipient_country as country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
        ) as affiliation
    ) as lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) as investigators,
    -- Source XML is the canonical landing surface IDRC publishes
    a.source_xml_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(a.iati_identifier)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM aggregated a
CROSS JOIN idrc_funder f
WHERE a.iati_identifier IS NOT NULL
  AND TRIM(a.iati_identifier) != '';

## Step 3: Insert into openalex_awards_raw at priority 37

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'idrc_iati' AND priority = 37;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    37 as priority
FROM openalex.awards.idrc_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_idrc_awards FROM openalex.awards.idrc_awards;

In [ ]:
%sql
-- Step 6.7 fail-fast amount/currency check (per how-to-add-a-funder.md)
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.idrc_awards;

In [ ]:
%sql
-- Sample for visual review
SELECT id, SUBSTRING(display_name, 1, 80) as title, funder_award_id,
       amount, currency, start_date, end_date,
       lead_investigator.affiliation.name as recipient_org,
       lead_investigator.affiliation.country as recipient_country
FROM openalex.awards.idrc_awards LIMIT 10;

In [ ]:
%sql
-- Year distribution
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_funding_cad
FROM openalex.awards.idrc_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;

In [ ]:
%sql
-- Top recipient countries
SELECT lead_investigator.affiliation.country as country, COUNT(*) as cnt
FROM openalex.awards.idrc_awards
WHERE lead_investigator.affiliation.country IS NOT NULL
GROUP BY country ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
-- Funder consistency
SELECT funder.display_name, COUNT(*) FROM openalex.awards.idrc_awards GROUP BY funder.display_name;